## 1. Basic Tasks

1. Create a notebook and add a markdown cell summarizing, in your own words, the difference
between a warehouse, a lake, and a lakehouse.

### Warehouse:
Warehouse is like a data center where we store structured and processed data. In this, we traditionally use ETL, which means Extract, Transform, and Load. First, we extract the data from different sources, then we do transformation, which means cleaning the data, like removing duplicates, dropping or filling null values, correcting data types, etc., so that the data can be used directly for analysis and reporting. But it has some disadvantages, like we need to do the transformation before loading the data, which can make it difficult to use the same data for different purposes later. If we don't store the original raw data, we may lose the flexibility to apply different transformations in the future, which can be a limitation for some AI and ML use cases that may need raw data. Also, traditional warehouses mainly focus on structured data, so they are less flexible when we need to work with unstructured data like images, videos, audio, or documents. Additionally, the transformation process requires compute resources, which can increase the overall cost depending on the architecture.

### Data Lake:
Data Lake is like a place where we store structured, semi-structured, and unstructured data in its raw form. It usually follows ELT, which means Extract, Load, and Transform. First, we extract the data from different sources and load it into the data lake, and then we transform the data whenever it is required. Because it stores raw data, it can be useful in many situations, such as AI/ML, where we may need the original data for different types of analysis and transformations. It is also generally cheaper than a traditional data warehouse for large-scale storage because it uses low-cost storage and separates storage from compute. However, one of the disadvantages of a traditional data lake is that it does not inherently provide ACID transactions, which can cause problems with data consistency when multiple users or pipelines are writing and modifying data at the same time. Because of these problems, poorly managed data lakes can sometimes become “data swamps,” where the data is difficult to find, understand, trust, and use.

### Lakehouse:
Lakehouse is a combination of both Data Lake and Data Warehouse, so it takes the advantages of both. Like a Data Lake, it can store structured, semi-structured, and unstructured data, including raw data, which makes it useful for AI/ML use cases. Like a Data Warehouse, it provides features such as ACID transactions, which help maintain data consistency, and it maintains transaction log files that track changes to the data. Because of these features, it reduces the chances of a data lake becoming a data swamp. It can be used for many different use cases, including data analytics, BI, AI, and ML, while still providing flexibility to transform the raw data whenever required. It can also be cost-effective because it separates storage and compute, allowing compute resources to be used only when needed.


2. Create a Delta table from ~10 rows of sample product data (product_id, name, category, price).

In [0]:
%sql
CREATE TABLE dev.bronze.products (
    product_id INT,
    name STRING,
    catagory STRING,
    price DOUBLE
)

In [0]:
%sql
INSERT INTO dev.bronze.products (product_id, name, catagory, price) VALUES
    (1, 'Laptop', 'Electronics', 74999.00),
    (2, 'Smartphone', 'Electronics', 29999.00),
    (3, 'Headphones', 'Electronics', 2499.00),
    (4, 'Office Chair', 'Furniture', 8999.00),
    (5, 'Desk Lamp', 'Furniture', 1499.00),
    (6, 'Backpack', 'Accessories', 1999.00),
    (7, 'Running Shoes', 'Footwear', 3499.00),
    (8, 'Wrist Watch', 'Accessories', 5999.00),
    (9, 'Coffee Maker', 'Kitchen', 4499.00),
    (10, 'Water Bottle', 'Kitchen', 799.00);

3. Run three separate INSERT/UPDATE statements against the table, then use DESCRIBE HISTORY to
view the resulting versions.

In [0]:
%sql
UPDATE dev.bronze.products SET price = price * 1.5 WHERE catagory = "Furniture" 

In [0]:
%sql
INSERT INTO dev.bronze.products (product_id, name, catagory, price) VALUES
    (11, 'Tablet', 'Electronics', 39999.00),
    (12, 'Monitor', 'Electronics', 59999.00)

In [0]:
%sql
UPDATE dev.bronze.products SET price = price - (price * 0.2) WHERE catagory = "Electronics" 

In [0]:
%sql
DESC HISTORY dev.bronze.products;

4. (Data Analyst) Use SELECT ... VERSION AS OF to query an older version of the table and note what
changed between versions.

In [0]:
%sql
SELECT * FROM dev.bronze.products VERSION AS OF 2

In [0]:
%sql
SELECT * FROM dev.bronze.products VERSION AS OF 4

- In the version 2 it includes the first update command so it only shows updated values of the furniture catagory and it only includes 10 rows.
- In the version 4 it includes 12 rows as i have used insert into statement in that version and the updated values of the furniture category.

## 2. Intermediate Tasks

5. Deliberately insert a row with an extra column and observe Delta's schema enforcement rejecting it;
then re-insert using mergeSchema and confirm schema evolution succeeded.

In [0]:
%sql
INSERT INTO dev.bronze.products VALUES 
    (13, 'Laptop', 'Electronics', 79999.00, '2026-08-20')

In [0]:
from datetime import date

schema = ("product_id int, name string, catagory string, price double, ingestion_date date")
data = [(13, 'Laptop', 'Electronics', 79999.00, date(2026, 8, 20))]

df1 = spark.createDataFrame(data, schema)

df1.write.mode("append").option("mergeSchema", True).saveAsTable("dev.bronze.products")

6. Use time travel (VERSION AS OF and TIMESTAMP AS OF) to reconstruct the table as it looked before a
simulated bad update, then write the RESTORE command that would fix it.

In [0]:
%sql
DESC HISTORY dev.bronze.products

In [0]:
%sql
SELECT * FROM dev.bronze.products VERSION AS OF 7

In [0]:
%sql
SELECT * FROM dev.bronze.products TIMESTAMP AS OF "2026-08-20T12:36:20.000+00:00"

In [0]:
%sql
SELECT * FROM dev.bronze.products VERSION AS OF 6

In [0]:
%sql
RESTORE TABLE dev.bronze.products TO VERSION AS OF 6

7. Write a short explanation, aimed at a non-technical stakeholder, of why ACID transactions matter
when multiple pipelines write to the same table concurrently.

In ACID A(Atomicity) means all or noting, C(Consistency) means that befor and after the transaction the net value should be same, I(Isolation) means concurrent trunsaction does not affect each other and D(Durability) means after the transaction gets commited, then it gets purmanently stored.

So when multiple pipelines write to the same table at the same time or concurrently then also it also make sure that data updates are handled properly means that either the changes are fully commited or doesn't get commited at all. The table remains valid before and after the transaction. No two changes affect each other. Once commited the changes are saved it won't be lost.

## 3. Advanced Tasks

8. Write a short design note describing how Cyntexa could replace a nightly batch warehouse load with
a lakehouse pipeline, calling out specifically where ACID transactions and time travel reduce

We can replace the Cyntexa's nightly batch update with lakehouse pipelines by when data came from different sources so instead of waiting and update nightly we can can update it concurrently or in small batches and with very small gaps between them, so we get the data in he bronze layer almost continiously, after that we can clean and transform the data and insert it into silver layer and after that according to the business needs we can create the views and use it for BI and dashboard.

ACID transactions ensure that either the changes are fully commited or doesn't get commited at all. The table remains valid before and after the transaction. No two changes affect each other. Once commited the changes are saved it won't be lost.
Time Travel provides us a way to go back to the previous versions in case of any wrong write query, so it help us to troubleshoot and makes it easy for us to recover table data of a previous versions of the table.
Overall, the lakehouse approach provides safer concurrent processing, easier recovery, and better traceability than a single nightly warehouse load.

9. Simulate two concurrent writers appending to the same Delta table (two notebook cells or jobs),
then use DESCRIBE HISTORY to explain how the transaction log resolved the write order and what
would happen if the writes conflicted.

In [0]:
%sql
DESC HISTORY dev.bronze.products

In [0]:
%sql
SELECT * from dev.bronze.products VERSION AS OF 9

In [0]:
%sql
SELECT * from dev.bronze.products VERSION AS OF 10

In [0]:
%sql
DESC HISTORY dev.bronze.products

In [0]:
%sql
SELECT * FROM dev.bronze.products VERSION AS OF 12

I have created two note books task-1 and task-2 in the this folder and the in task-1 I have done this 

UPDATE dev.bronze.products SET price = 55000.00 WHERE product_id = 1;

and in task-2 I have done this 

UPDATE dev.bronze.products SET price = 55000.00 WHERE product_id = 1;

and then I have created two jobs for those two tasks job-1 and job-2 and scheduled at the same time but still there is second time difference so both of those tasks gets executed as we can see the desc history part.
But later I have created a job named concurrent_job where i have put both the task-1 and task-2 in the same job parallelly and run the job then the task I have created first in the job gets executed and the second task gets blocked and gets this error:
**" Transaction conflict detected. A concurrent UPDATE added data to table dev.bronze.products committed at version 12. The concurrent operation modified the same rows that this transaction attempted to modify. "**

10. (Data Analyst) Write a one-page comparison memo: list 3 concrete advantages a lakehouse gives
analysts over a traditional warehouse, and 1 tradeoff to watch for.

A lakehouse is a combination of both data warehouse and data lake, It takes the advantages of a traditional warehouse of ACID and all and also the flexibility of storing different types of data like structured, semi-structured, and unstructured data.

**1. Can store multiple types of data**

A lakehouse can store multiple types of data like structured, semi-structured, and unstructured data so we can use it for many use cases like where raw data is needed it can provide raw data it AI and ML use cases, also where it needs structured data we can get structed data as well and as it is not transformed before hand you can do according to your need.

**2. Usefull for incremental and streaming pipelines and historical data**

Warehouse depends on batch streaming but by using lakehouse we can do incremental and streaming as it supports both pipelines.
Also here analysts gets to work on raw historincal data instead of highly processed or transformed data that is stored in the data warehouse as it follows ETL.

**3. Better Data Reliability and Recovery**

Lakehouse technologies such as Delta Lake provide ACID transactions and features such as Time Travel. Time Travel allows analysts or data engineers to examine previous versions of a table. For example, if a data pipeline accidentally changes or deletes records, an earlier version can be inspected or restored. This makes troubleshooting and validating historical results easier.

**Tradeoff: Complexity**

The main tradeoff of Lakehouse is complexity as for lakehouse it enforce strong governance and unity catalog features, and also for every query you get different log files and paraquete files and to make sure everything works fine. 
So it introduce more complexity along with all the benifits it provides over the warehouse and data lake.

**Conclusion**

For analysts, a lakehouse offers greater data variety, stronger reliability and recovery capabilities. However, these benefits come with additional platform and governance complexity. With appropriate standards and tooling, the flexibility and reliability of a lakehouse can make it a strong alternative to a traditional warehouse.
